# Sentinel-1 L0 → SLC On-Demand Processing

Orders the processing of a Sentinel-1 Level-0 product to SLC using the ESA Copernicus Data Space
On-Demand Production API (`Sentinel-1-L0-EW_SLC__1S` workflow).

**Workflow:**
1. Authenticate with Copernicus Identity Service
2. Confirm the workflow exists
3. Submit a production order for a specified L0 product
4. Poll until the order completes
5. Download the resulting SLC `.zip`

**References:**
- [On-Demand Production API docs](https://documentation.dataspace.copernicus.eu/APIs/On-Demand%20Production%20API.html)
- [OData API](https://odp.dataspace.copernicus.eu/odata/docs)
- [Copernicus Data Space](https://dataspace.copernicus.eu)

## Configuration

Set the L0 product name and your output directory below. Credentials are read from environment
variables `CDSE_USERNAME` and `CDSE_PASSWORD` (or enter them directly — avoid committing passwords).

In [ ]:
import os
from dotenv import load_dotenv

try:
    # load credentials from root .env file
    load_dotenv("../.env")
except:
    print('Could not find .env file with credentials.')

# --- INPUT -------------------------------------------------------------------
# Name of the Level-0 EW product to process (without trailing slash)
L0_PRODUCT = "S1D_EW_RAW__0SSH_20260601T154450_20260601T154558_003044_005414_5569.SAFE"

# Where to save the downloaded SLC zip
OUTPUT_DIR = "."

# A short label for the order (visible in the ODP portal)
ORDER_NAME = f"slc_from_{L0_PRODUCT[:32]}"

# Credentials — prefer env vars so they are not committed
CDSE_USERNAME = os.environ.get("CDSE_LOGIN", "")   # or set directly: "your@email.com"
CDSE_PASSWORD = os.environ.get("CDSE_PASSWORD", "")   # or set directly: "yourpassword"
# -----------------------------------------------------------------------------

WORKFLOW_NAME = "Sentinel-1-L0-EW_SLC__1S"
BASE_URL      = "https://odp.dataspace.copernicus.eu/odata/v1"
TOKEN_URL     = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

print(f"L0 product : {L0_PRODUCT}")
print(f"Workflow   : {WORKFLOW_NAME}")
print(f"Output dir : {os.path.abspath(OUTPUT_DIR)}")

## 1. Authenticate

In [ ]:
import requests
import getpass
import time as _time

if not CDSE_USERNAME:
    CDSE_USERNAME = input("Copernicus username (email): ")
if not CDSE_PASSWORD:
    CDSE_PASSWORD = getpass.getpass("Copernicus password: ")

def get_token(username: str, password: str) -> str:
    resp = requests.post(
        TOKEN_URL,
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise RuntimeError(
            f"Authentication failed ({resp.status_code}): {detail}\n\n"
            "Check that:\n"
            "  1. Your Copernicus Data Space account is verified at https://dataspace.copernicus.eu\n"
            "  2. You are using your registration email + password (not a Google/GitHub SSO login)\n"
            "  3. There are no leading/trailing spaces in your credentials"
        )
    data = resp.json()
    return data["access_token"], data.get("expires_in", 600)

TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
TOKEN_ACQUIRED_AT = _time.monotonic()

def auth_headers() -> dict:
    """Return headers with a fresh token, refreshing if within 60s of expiry."""
    global TOKEN, TOKEN_ACQUIRED_AT, TOKEN_EXPIRES_IN
    if _time.monotonic() - TOKEN_ACQUIRED_AT > (TOKEN_EXPIRES_IN - 60):
        print("Refreshing token...")
        TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
        TOKEN_ACQUIRED_AT = _time.monotonic()
    return {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

AUTH_HEADERS = auth_headers()
print(f"Authenticated successfully. Token valid for {TOKEN_EXPIRES_IN}s.")

## 2. Confirm Workflow Exists

In [ ]:
import json

resp = requests.get(
    f"{BASE_URL}/Workflows",
    params={"$filter": f"Name eq '{WORKFLOW_NAME}'"},
    headers=auth_headers(),
    timeout=30,
)
resp.raise_for_status()
workflows = resp.json().get("value", [])

if not workflows:
    raise RuntimeError(
        f"Workflow '{WORKFLOW_NAME}' not found. "
        "Check available workflows at: GET /odata/v1/Workflows"
    )

wf = workflows[0]
print(json.dumps(wf, indent=2))

## 3. Submit Production Order

In [ ]:
order_payload = {
    "WorkflowName": WORKFLOW_NAME,
    "InputProductReference": {
        "Reference": L0_PRODUCT,
    },
    "WorkflowOptions": [
        # Uncomment / add options as needed for this processor.
        # e.g. {"Name": "output_storage", "Value": "TEMPORARY"}
    ],
    "Priority": 1,
    "Name": ORDER_NAME,
}

resp = requests.post(
    f"{BASE_URL}/ProductionOrder/OData.CSC.Order",
    headers=auth_headers(),
    json=order_payload,
    timeout=60,
)
resp.raise_for_status()
order = resp.json()['value']

ORDER_ID = order["Id"]
print(f"Order submitted.")
print(f"  ID     : {ORDER_ID}")
print(f"  Status : {order.get('Status')}")
print(f"  Name   : {order.get('Name')}")

## 4. Poll Until Complete

Processing typically takes several minutes. The cell below polls every 60 seconds and prints
a status update. Interrupt the kernel if you want to stop polling and resume manually.

In [ ]:
import time
from datetime import datetime

POLL_INTERVAL_SECONDS = 60
TERMINAL_STATUSES = {"completed", "failed", "cancelled"}

def get_order_status(order_id: str) -> dict:
    r = requests.get(
        f"{BASE_URL}/ProductionOrders({order_id})",
        headers=auth_headers(),   # refreshes token automatically
        timeout=30,
    )
    if not r.ok:
        print(f"  Warning: {r.status_code} — {r.text[:200]}")
        r.raise_for_status()
    return r.json()['value']  # single-item response, no "value" wrapper

while True:
    order_info = get_order_status(ORDER_ID)
    status = order_info.get("Status", "unknown").lower()
    print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC]  Status: {status}")

    if status in TERMINAL_STATUSES:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

if status != "completed":
    print("\nFull order details:")
    print(json.dumps(order_info, indent=2))
    raise RuntimeError(f"Order ended with status '{status}' — cannot download.")

print("\nOrder completed — ready to download.")

## 5. Inspect Output Product

In [ ]:
resp = requests.get(
    f"{BASE_URL}/ProductionOrder({ORDER_ID})/Product",
    headers=AUTH_HEADERS,
    timeout=30,
)
resp.raise_for_status()
product_info = resp.json()
print(json.dumps(product_info, indent=2))

# Derive a sensible local filename from the product name if available
product_name = product_info.get("Name", f"order_{ORDER_ID}_slc")
if not product_name.endswith(".zip"):
    product_name += ".zip"
LOCAL_PATH = os.path.join(OUTPUT_DIR, product_name)
print(f"\nWill save to: {os.path.abspath(LOCAL_PATH)}")

## 6. Download SLC Product

In [ ]:
from tqdm.notebook import tqdm

os.makedirs(OUTPUT_DIR, exist_ok=True)

download_url = f"{BASE_URL}/ProductionOrder({ORDER_ID})/Product/$value"

with requests.get(download_url, headers=AUTH_HEADERS, stream=True, timeout=300) as r:
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))
    with open(LOCAL_PATH, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc=os.path.basename(LOCAL_PATH)
    ) as bar:
        for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
            f.write(chunk)
            bar.update(len(chunk))

size_mb = os.path.getsize(LOCAL_PATH) / 1e6
print(f"\nDownloaded {size_mb:.1f} MB → {os.path.abspath(LOCAL_PATH)}")

## 7. (Optional) List All Your Orders

Handy for checking the status of past orders or finding `ORDER_ID` values.

In [ ]:
resp = requests.get(
    f"{BASE_URL}/ProductionOrders",
    params={"$filter": f"WorkflowName eq '{WORKFLOW_NAME}'", "$orderby": "SubmissionDate desc", "$top": 10},
    headers=AUTH_HEADERS,
    timeout=30,
)
resp.raise_for_status()
orders = resp.json().get("value", [])

for o in orders:
    print(f"{o['Id']}  {o.get('Status', '?'):15s}  {o.get('SubmissionDate', '?')}  {o.get('Name', '')}")

## 8. (Optional) Cancel an Order

Uncomment and run if you need to cancel a queued or in-progress order.

In [ ]:
# CANCEL_ORDER_ID = ORDER_ID  # or paste a specific ID
# resp = requests.delete(
#     f"{BASE_URL}/ProductionOrder({CANCEL_ORDER_ID})/OData.CSC.Cancel",
#     headers=AUTH_HEADERS,
#     timeout=30,
# )
# resp.raise_for_status()
# print(f"Cancelled order {CANCEL_ORDER_ID}")

## Errors

- After a few weeks L0 products might be pulled from long term archive (LTA) storage and might not be available
{
  "Id": 10661506,
  "Status": "failed",
  "StatusMessage": "input product not found on LTA",
  "SubmissionDate": "2026-06-03T00:03:30.589Z",
  "Name": "slc_from_S1C_EW_RAW__0SSH_20250601T175535",
  "EstimatedDate": "2026-06-03T02:41:13.589Z",
  "InputProductReference": {
    "Reference": "S1C_EW_RAW__0SSH_20250601T175535_20250601T175613_002592_00560A_B642.SAFE",
    "ContentDate": null
  },
  "WorkflowOptions": [
    {
      "Name": "platform",
      "Value": "cdse"
    },
    {
      "Name": "version",
      "Value": "4.0.3"
    },
    {
      "Name": "output_storage",
      "Value": "TEMPORARY"
    }
  ],
  "WorkflowName": "Sentinel-1-L0-EW_SLC__1S",
  "WorkflowId": 102,
  "Priority": 1
}